# Interface location (`z_dis`) error comparison using `Solver`

Runs the two-layer FD solver for multiple `z_dis` values on a **fixed grid**.
The first entry in `z_dis_list` is the **base/reference** solution.
All other runs are compared against it pointwise.

Because `Lz` and `nz` are fixed, all solutions share the same `z` grid —
no interpolation is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.set_default_dtype(torch.float64)

In [ ]:
%run func_defs.ipynb
%run 2L_NN_v7.ipynb

## Parameters

Edit `z_dis_list` to add or remove interface locations.
The **first** entry is the reference (`z_dis = 1.0` by default).

In [ ]:
# ── shared physics ──────────────────────────────────────────────────────
k1      = 1.0
k2      = 2.0
Pda     = 1.0
tf      = 2.0
Lz      = 16.0          # fixed domain for all runs

# ── grid ─────────────────────────────────────────────────────────────────
nz_unit = 10            # grid points per unit length
nz      = int(Lz) * nz_unit
n_t_out = 80

# ── interface locations (first = reference) ───────────────────────────────
z_dis_list = [1.0, 2.0, 4.0, 8.0]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device},  Lz={Lz},  nz={nz}')

## Run `Solver` for each `z_dis`

In [ ]:
solvers = {}   # z_dis -> Solver instance

for z_dis in z_dis_list:
    print(f'\n=== z_dis={z_dis} ===')
    solvers[z_dis] = Solver(
        k1=k1, k2=k2, z_dis=z_dis, Pda=Pda,
        Lz=Lz, nz=nz, tf=tf, n_t_out=n_t_out,
        device=device, verbose=True,
    )
    print(f'  U shape: {solvers[z_dis]["U"].shape}')

## Compute errors vs the reference (`z_dis_ref`)

All grids are identical so no interpolation is required.

In [ ]:
z_dis_ref = z_dis_list[0]          # reference: z_dis = 1.0
U_ref     = solvers[z_dis_ref]['U']  # (nz+1, n_t_out)
z_grid    = solvers[z_dis_ref]['z']  # shared z grid
t_out     = solvers[z_dis_ref]['t']  # shared time grid

errors_abs = {}   # z_dis -> (nz+1, n_t_out)
errors_rel = {}

for z_dis in z_dis_list[1:]:
    diff = (solvers[z_dis]['U'] - U_ref).abs()
    errors_abs[z_dis] = diff
    errors_rel[z_dis] = diff / (U_ref.abs() + 1e-40)

# summary table
print(f'  {"z_dis":>8}  {"max |err|":>12}  {"mean |err|":>12}  {"max rel err":>12}')
print('-' * 52)
for z_dis, err in errors_abs.items():
    rel = errors_rel[z_dis]
    print(f'  {z_dis:>8.2f}  {err.max().item():>12.4e}'
          f'  {err.mean().item():>12.4e}  {rel.max().item():>12.4e}')

## Plots

In [ ]:
n_runs  = len(z_dis_list)
idx     = torch.linspace(0, n_t_out - 1, min(6, n_t_out)).long()
t_cpu   = t_out.cpu()
z_cpu   = z_grid.cpu()

fig, axes = plt.subplots(1, n_runs, figsize=(6 * n_runs, 4), sharey=True)
if n_runs == 1:
    axes = [axes]

for ax, z_dis in zip(axes, z_dis_list):
    U_plot = solvers[z_dis]['U'].cpu()
    for i in idx:
        ax.plot(z_cpu, U_plot[:, i], label=f't={t_cpu[i]:.3f}')
    ax.axvline(z_dis, color='r', linestyle='--', lw=1.0, label=f'z_dis={z_dis}')
    tag = ' [ref]' if z_dis == z_dis_ref else ''
    ax.set_xlabel('z'); ax.set_ylabel('U')
    ax.set_title(f'z_dis={z_dis}{tag}')
    ax.legend(fontsize=7)

plt.suptitle('Smoothed U(z,t) for each interface location', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
n_err = len(errors_abs)
fig, axes = plt.subplots(1, n_err, figsize=(6 * n_err, 4), sharey=True)
if n_err == 1:
    axes = [axes]

for ax, (z_dis, err) in zip(axes, errors_abs.items()):
    err_cpu = err.cpu()
    for i in idx:
        ax.plot(z_cpu, err_cpu[:, i], label=f't={t_cpu[i]:.3f}')
    ax.axvline(z_dis,     color='r', linestyle='--', lw=1.0, label=f'z_dis={z_dis}')
    ax.axvline(z_dis_ref, color='k', linestyle=':',  lw=0.8, label=f'z_dis_ref={z_dis_ref}')
    ax.set_xlabel('z'); ax.set_ylabel('|ΔU|')
    ax.set_title(f'|U(z_dis={z_dis}) − U(z_dis={z_dis_ref})|')
    ax.legend(fontsize=7)

plt.suptitle(f'Absolute error vs reference (z_dis={z_dis_ref})', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, n_err, figsize=(6 * n_err, 4), sharey=True)
if n_err == 1:
    axes = [axes]

for ax, (z_dis, rel) in zip(axes, errors_rel.items()):
    rel_cpu = rel.cpu()
    for i in idx:
        ax.semilogy(z_cpu, rel_cpu[:, i].clamp(min=1e-16), label=f't={t_cpu[i]:.3f}')
    ax.axvline(z_dis,     color='r', linestyle='--', lw=1.0, label=f'z_dis={z_dis}')
    ax.axvline(z_dis_ref, color='k', linestyle=':',  lw=0.8, label=f'z_dis_ref={z_dis_ref}')
    ax.set_xlabel('z'); ax.set_ylabel('relative |ΔU|')
    ax.set_title(f'Relative error: z_dis={z_dis} vs z_dis={z_dis_ref}')
    ax.legend(fontsize=7)

plt.suptitle(f'Relative error vs reference (z_dis={z_dis_ref})', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for z_dis, err in errors_abs.items():
    max_err_t = err.max(dim=0).values.cpu()
    ax.semilogy(t_cpu, max_err_t, label=f'z_dis={z_dis}')

ax.set_xlabel('t')
ax.set_ylabel('max_z |ΔU|')
ax.set_title(f'Max absolute error over z vs time  (ref z_dis={z_dis_ref})')
ax.legend()
plt.tight_layout()
plt.show()